# 📦 Tutorial Bonus: Trabajando con Datasets Reales

## De Datos Sintéticos a Benchmarks Estándar

En este tutorial aprenderás:

- 📊 Cómo cargar y preparar datasets reales de Few-Shot Learning
- 🎯 Omniglot: El "MNIST" del Meta-Learning
- 🖼️ Mini-ImageNet: El benchmark estándar
- 🔄 Crear samplers N-way K-shot apropiados
- 📈 Protocolo de evaluación estándar

---

## 📖 Parte 1: Introducción a Datasets de Few-Shot

### ¿Por qué Datasets Especiales?

En Meta-Learning necesitamos datasets con:
- **Muchas clases** (para tener suficientes tareas)
- **Pocos ejemplos por clase** (para simular few-shot)
- **Split especial**: train/val/test a nivel de CLASES, no ejemplos

### Datasets Estándar:

#### 1. **Omniglot** 📝
- 1,623 caracteres de 50 alfabetos diferentes
- 20 ejemplos por clase
- Imágenes 28x28 en escala de grises
- **Uso**: Baseline rápido, prueba de concepto

#### 2. **Mini-ImageNet** 🖼️
- 100 clases de ImageNet
- 600 ejemplos por clase
- Imágenes 84x84 RGB
- **Uso**: Benchmark estándar principal

#### 3. **Tiered-ImageNet** 📚
- 608 clases organizadas jerárquicamente
- Más desafiante que Mini-ImageNet

### Split a Nivel de Clases:

```
Dataset Total: 100 clases
├── Train: 64 clases (para meta-training)
├── Val: 16 clases (para validación/tuning)
└── Test: 20 clases (para evaluación final)

⚠️ IMPORTANTE: Las clases NO se solapan!
```


## 📑 Table of Contents- [1 - Introduction to Few-Shot Datasets](#1)    - [1.1 - Why Special Datasets?](#1-1)    - [1.2 - Standard Benchmarks](#1-2)- [2 - Setup and Downloads](#2)- [3 - Omniglot Dataset](#3)    - [3.1 - Loading and Exploration](#3-1)    - [3.2 - Data Augmentation](#3-2)- [4 - Exercise 1 - N-way K-shot Sampler](#ex-1)- [5 - Exercise 2 - Episode Visualization](#ex-2)- [6 - Standard Evaluation Protocol](#6)- [7 - Training with Real Data](#7)- [8 - Benchmarking Results](#8)- [9 - Comparison: Synthetic vs Real Data](#9)- [10 - Mini-ImageNet Preview](#10)- [11 - Summary](#11)

---

## 🛠️ Parte 2: Setup

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
from collections import defaultdict
import random
import sys
sys.path.append('..')

from utils.test_utils import print_success, print_hint, HintSystem
from utils.data_utils import set_seed

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Setup completo!")

---

## 📊 Parte 3: Descargar Omniglot

Vamos a usar torchvision para descargar Omniglot fácilmente.

In [ ]:
# Descargar Omniglot
print("📦 Descargando Omniglot...\n")

# Background set (para training)
omniglot_train = torchvision.datasets.Omniglot(
    root='../datasets',
    background=True,
    download=True,
    transform=transforms.Compose([
        transforms.Resize((28, 28)),
        transforms.ToTensor(),
    ])
)

# Evaluation set (para test)
omniglot_test = torchvision.datasets.Omniglot(
    root='../datasets',
    background=False,
    download=True,
    transform=transforms.Compose([
        transforms.Resize((28, 28)),
        transforms.ToTensor(),
    ])
)

print(f"✅ Omniglot descargado!")
print(f"  Train: {len(omniglot_train)} imágenes")
print(f"  Test: {len(omniglot_test)} imágenes")

In [ ]:
# Visualizar algunos ejemplos
fig, axes = plt.subplots(4, 10, figsize=(15, 6))

for i in range(40):
    idx = random.randint(0, len(omniglot_train) - 1)
    img, label = omniglot_train[idx]
    
    ax = axes[i // 10, i % 10]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.axis('off')

plt.suptitle('Ejemplos de Omniglot', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("🎨 Cada imagen es un carácter único de diferentes alfabetos del mundo.")

---

## 💻 Parte 4: Ejercicio - N-way K-shot Sampler

El corazón de Few-Shot Learning: crear episodios de entrenamiento.

**Tu tarea**: Implementa un sampler que cree tareas N-way K-shot.

In [ ]:
class OmniglotNWayKShot:
    """
    Sampler para crear episodios N-way K-shot de Omniglot.
    """
    
    def __init__(self, dataset, n_way=5, k_shot=1, q_query=15):
        """
        Args:
            dataset: Dataset de Omniglot
            n_way: Número de clases por episodio
            k_shot: Ejemplos de soporte por clase
            q_query: Ejemplos de query por clase
        """
        self.dataset = dataset
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        
        # TODO: Organiza el dataset por clases
        # Crea un diccionario: {class_id: [lista de índices de esa clase]}
        self.class_to_indices = self._organize_by_class()
        self.classes = list(self.class_to_indices.keys())
    
    def _organize_by_class(self):
        """
        Organiza los índices del dataset por clase.
        
        Returns:
            Dict[int, List[int]]: Mapa de clase a lista de índices
        """
        # TODO: Implementa esta función
        # Recorre todo el dataset y agrupa los índices por su label
        
        pass  # TODO: Elimina y escribe tu código
    
    def sample_episode(self):
        """
        Genera un episodio N-way K-shot.
        
        Returns:
            support_x: [n_way * k_shot, C, H, W]
            support_y: [n_way * k_shot]
            query_x: [n_way * q_query, C, H, W]
            query_y: [n_way * q_query]
        """
        # TODO: Implementa el sampling de episodios
        # 1. Selecciona n_way clases aleatorias
        # 2. Para cada clase, selecciona k_shot + q_query ejemplos
        # 3. Los primeros k_shot van a support, el resto a query
        # 4. Crea tensores y retorna
        
        pass  # TODO: Elimina y escribe tu código


# Sistema de pistas
hints_sampler = HintSystem([
    "Para _organize_by_class: usa defaultdict(list) y recorre el dataset con enumerate().",
    "Para cada (idx, (img, label)) en el dataset, agrega idx a class_to_indices[label].",
    "Para sample_episode: usa random.sample(self.classes, n_way) para elegir clases.",
    "Para cada clase, usa random.sample(class_indices, k_shot + q_query) para elegir ejemplos."
])

In [ ]:
# Para ver pistas
hints_sampler.show_hint()

In [ ]:
# ✅ TEST: Verificar sampler

sampler = OmniglotNWayKShot(omniglot_train, n_way=5, k_shot=1, q_query=15)

support_x, support_y, query_x, query_y = sampler.sample_episode()

print(f"📊 Episodio generado:")
print(f"  Support: {support_x.shape}, Labels: {support_y.shape}")
print(f"  Query: {query_x.shape}, Labels: {query_y.shape}")
print(f"  Clases únicas en support: {torch.unique(support_y).tolist()}")
print(f"  Clases únicas en query: {torch.unique(query_y).tolist()}")

assert support_x.shape[0] == 5, "Support debe tener 5 ejemplos (5-way 1-shot)"
assert query_x.shape[0] == 75, "Query debe tener 75 ejemplos (5 clases * 15 queries)"
assert len(torch.unique(support_y)) == 5, "Debe haber 5 clases únicas"

print_success("\n✅ Sampler funciona correctamente!")

---

## 🎨 Parte 5: Visualizar un Episodio

In [ ]:
# Visualizar un episodio 5-way 5-shot
sampler = OmniglotNWayKShot(omniglot_train, n_way=5, k_shot=5, q_query=5)
support_x, support_y, query_x, query_y = sampler.sample_episode()

fig, axes = plt.subplots(5, 10, figsize=(15, 8))

for class_idx in range(5):
    # Support examples (primeros 5)
    support_mask = support_y == class_idx
    support_imgs = support_x[support_mask]
    
    for i in range(5):
        ax = axes[class_idx, i]
        ax.imshow(support_imgs[i].squeeze(), cmap='gray')
        ax.axis('off')
        if i == 0:
            ax.set_title(f'Clase {class_idx}', fontsize=10, fontweight='bold')
        if class_idx == 0:
            ax.text(0.5, 1.15, 'SUPPORT', transform=ax.transAxes,
                   ha='center', fontsize=9, color='blue')
    
    # Query examples (siguientes 5)
    query_mask = query_y == class_idx
    query_imgs = query_x[query_mask]
    
    for i in range(5):
        ax = axes[class_idx, i + 5]
        ax.imshow(query_imgs[i].squeeze(), cmap='gray')
        ax.axis('off')
        if class_idx == 0:
            ax.text(0.5, 1.15, 'QUERY', transform=ax.transAxes,
                   ha='center', fontsize=9, color='green')

plt.suptitle('Episodio 5-way 5-shot', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📝 Cada fila es una clase diferente.")
print("   Azul (SUPPORT): Ejemplos para adaptar el modelo")
print("   Verde (QUERY): Ejemplos para evaluar la adaptación")

---

## 📈 Parte 6: Protocolo de Evaluación Estándar

¿Cómo reportar resultados en papers?

In [ ]:
def evaluate_few_shot(model, sampler, n_episodes=600):
    """
    Evaluación estándar de Few-Shot Learning.
    
    Protocolo:
    - Samplear n_episodes episodios del conjunto de test
    - Para cada episodio: adaptar y evaluar
    - Reportar: media ± intervalo de confianza 95%
    
    Args:
        model: Modelo a evaluar
        sampler: Sampler de episodios
        n_episodes: Número de episodios (típicamente 600-1000)
    
    Returns:
        mean_acc: Accuracy promedio
        ci95: Intervalo de confianza 95%
    """
    model.eval()
    accuracies = []
    
    for episode in range(n_episodes):
        support_x, support_y, query_x, query_y = sampler.sample_episode()
        
        # Mover a device
        support_x = support_x.to(device)
        support_y = support_y.to(device)
        query_x = query_x.to(device)
        query_y = query_y.to(device)
        
        with torch.no_grad():
            # Forward pass (depende del modelo)
            logits = model(support_x, support_y, query_x, n_way=sampler.n_way)
            predictions = logits.argmax(dim=1)
            accuracy = (predictions == query_y).float().mean().item()
        
        accuracies.append(accuracy)
    
    # Calcular estadísticas
    mean_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)
    ci95 = 1.96 * std_acc / np.sqrt(n_episodes)
    
    return mean_acc, ci95


print("✅ Función de evaluación estándar definida!")
print("\n📊 Cómo reportar resultados:")
print("   Accuracy: 95.2% ± 0.3%")
print("            ^^^^   ^^^^")
print("            media  CI95")

---

## 🎓 Resumen

### ✅ Lo que aprendiste:

1. **Omniglot** es el dataset de referencia para Few-Shot Learning
2. El **split es a nivel de clases**, no de ejemplos
3. Un **sampler N-way K-shot** crea episodios de entrenamiento
4. La **evaluación estándar** usa 600-1000 episodios y reporta media ± CI95

### 📊 Benchmarks Típicos en Papers:

**Omniglot:**
- 5-way 1-shot: ~98-99%
- 5-way 5-shot: ~99-99.5%
- 20-way 1-shot: ~95-97%

**Mini-ImageNet:**
- 5-way 1-shot: ~48-55%
- 5-way 5-shot: ~63-70%

### 🚀 Próximos Pasos:

Usa estos datasets y samplers en los otros tutoriales para entrenar con datos reales!

---

## 🎉 ¡Ahora puedes trabajar con datasets reales!


<a name='3-2'></a>### 3.2 - Data Augmentation for Few-Shot LearningData augmentation is crucial when you have limited examples!

In [ ]:
import torchvision.transforms as transforms# Define augmentation pipelinetrain_transform = transforms.Compose([    transforms.Resize((28, 28)),    transforms.RandomRotation(degrees=15),    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),    transforms.ToTensor(),])test_transform = transforms.Compose([    transforms.Resize((28, 28)),    transforms.ToTensor(),])# Load with augmentationomniglot_train_aug = torchvision.datasets.Omniglot(    root='../datasets',    background=True,    download=True,    transform=train_transform)omniglot_test_aug = torchvision.datasets.Omniglot(    root='../datasets',    background=False,    download=True,    transform=test_transform)print("✅ Datasets loaded with augmentation!")print(f"   Train: {len(omniglot_train_aug)} images (with augmentation)")print(f"   Test:  {len(omniglot_test_aug)} images (no augmentation)")# Visualize augmentation effectfig, axes = plt.subplots(2, 5, figsize=(15, 6))idx = np.random.randint(0, len(omniglot_train))original_img, label = omniglot_train[idx]# Show originalaxes[0, 0].imshow(original_img.squeeze(), cmap='gray')axes[0, 0].set_title('Original')axes[0, 0].axis('off')# Show augmented versionsfor i in range(1, 5):    aug_img, _ = omniglot_train_aug[idx]    axes[0, i].imshow(aug_img.squeeze(), cmap='gray')    axes[0, i].set_title(f'Augmented {i}')    axes[0, i].axis('off')# Second row with different imageidx2 = np.random.randint(0, len(omniglot_train))for i in range(5):    aug_img, _ = omniglot_train_aug[idx2]    axes[1, i].imshow(aug_img.squeeze(), cmap='gray')    axes[1, i].axis('off')plt.suptitle('Data Augmentation Examples', fontsize=16, fontweight='bold')plt.tight_layout()plt.show()print("\n💡 Augmentation helps the model learn rotation and translation invariance!")

<a name='9'></a>## 9 - Comparison: Synthetic vs Real DataLet's compare model performance on synthetic vs real data.

In [ ]:
from utils.data_utils import create_classification_taskprint("🔬 Training on Synthetic Data vs Real Data...\n")# 1. Model trained on SYNTHETIC dataprint("Training model on SYNTHETIC data...")model_synthetic = PrototypicalNetwork(input_channels=1, embedding_dim=64)synthetic_losses = []for episode in range(200):    task = create_classification_task(n_way=5, k_shot=5, q_query=15)        model_synthetic.train()    optimizer = optim.Adam(model_synthetic.parameters(), lr=0.001)    criterion = nn.CrossEntropyLoss()        logits = model_synthetic(        task['x_support'],        task['y_support'],        task['x_query'],        n_way=5    )        loss = criterion(logits, task['y_query'])    optimizer.zero_grad()    loss.backward()    optimizer.step()        synthetic_losses.append(loss.item())print(f"  Final loss: {np.mean(synthetic_losses[-20:]):.4f}")# 2. Model trained on REAL data (Omniglot)print("\nTraining model on REAL data (Omniglot)...")model_real = PrototypicalNetwork(input_channels=1, embedding_dim=64)sampler_real = OmniglotNWayKShot(omniglot_train, n_way=5, k_shot=5, q_query=15)real_losses = []for episode in range(200):    support_x, support_y, query_x, query_y = sampler_real.sample_episode()        model_real.train()    optimizer = optim.Adam(model_real.parameters(), lr=0.001)    criterion = nn.CrossEntropyLoss()        logits = model_real(support_x, support_y, query_x, n_way=5)    loss = criterion(logits, query_y)        optimizer.zero_grad()    loss.backward()    optimizer.step()        real_losses.append(loss.item())print(f"  Final loss: {np.mean(real_losses[-20:]):.4f}")# Compare learning curvesfig, ax = plt.subplots(figsize=(12, 6))ax.plot(synthetic_losses, alpha=0.3, color='blue', label='Synthetic (raw)')ax.plot(np.convolve(synthetic_losses, np.ones(20)/20, mode='valid'),       color='blue', linewidth=2, label='Synthetic (smoothed)')ax.plot(real_losses, alpha=0.3, color='green', label='Real (raw)')ax.plot(np.convolve(real_losses, np.ones(20)/20, mode='valid'),       color='green', linewidth=2, label='Real (smoothed)')ax.set_xlabel('Episode', fontsize=12)ax.set_ylabel('Loss', fontsize=12)ax.set_title('Learning Curves: Synthetic vs Real Data', fontsize=14)ax.legend()ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("\n📊 Key Observations:")print("  - Real data typically has higher initial loss (more complex)")print("  - Both converge, but real data may need more episodes")print("  - Real data better prepares model for actual deployment")

<a name='10'></a>## 10 - Mini-ImageNet PreviewMini-ImageNet is the most challenging standard benchmark for few-shot learning.### Dataset Details:- **100 classes** from ImageNet- **600 images per class**- **84x84 RGB images**- **Split**: 64 train / 16 val / 20 test classes### Loading Mini-ImageNet:```python# Note: Mini-ImageNet requires manual download# Download from: https://github.com/y2l/mini-imagenet-toolsfrom torchvision import datasets, transformsmini_imagenet_transform = transforms.Compose([    transforms.Resize((84, 84)),    transforms.ToTensor(),    transforms.Normalize(mean=[0.485, 0.456, 0.406],                       std=[0.229, 0.224, 0.225])])# Load dataset (after downloading)# mini_imagenet = datasets.ImageFolder(#     root='../datasets/mini-imagenet/train',#     transform=mini_imagenet_transform# )```### Expected Performance (from papers):<table><tr>    <td><b>Method</b></td>    <td><b>5-way 1-shot</b></td>    <td><b>5-way 5-shot</b></td></tr><tr>    <td>Prototypical Networks</td>    <td>49.42% ± 0.78%</td>    <td>68.20% ± 0.66%</td></tr><tr>    <td>Matching Networks</td>    <td>43.56% ± 0.84%</td>    <td>55.31% ± 0.73%</td></tr><tr>    <td>MAML</td>    <td>48.70% ± 1.84%</td>    <td>63.11% ± 0.92%</td></tr></table>**Note**: Mini-ImageNet is significantly harder than Omniglot due to:- Higher visual complexity- More intra-class variation- Color images (3 channels vs 1)